# Import libs

In [1]:
import os
import json
from pathlib import Path
import numpy as np
import torch


libgomp: Invalid value for environment variable OMP_NUM_THREADS


# With UNET

In [2]:
os.chdir('/home/jovyan/dnalm/')

In [3]:
from transformers import AutoTokenizer, AutoConfig
tokenizer = AutoTokenizer.from_pretrained('./data/tokenizers/t2t_1000h_multi_32k/')

/home/jovyan/dnalm/my_saved_conda_envs/gena/lib/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
from src.gena_lm.modeling_bert import BertForLetterLevelTokenClassification

In [5]:
model_cfg = AutoConfig.from_pretrained('./data/configs/L24-H1024-A16-V32k-preln-lastln.json')
model_cls = BertForLetterLevelTokenClassification
model = model_cls(config=model_cfg)

In [6]:
rmt_config = {
            'num_mem_tokens': 5,
            'max_n_segments': 10000,
            'input_size': 512,
            'bptt_depth': -1,
            'sum_loss': True,
            'tokenizer': tokenizer,
            'use_crf': False,
            'num_crf_classes': 6
        }
from src.gena_lm.modeling_rmt import RMTEncoderForLetterLevelTokenClassificationUNET
rmt_cls = RMTEncoderForLetterLevelTokenClassificationUNET
model_rmt = rmt_cls(model, **rmt_config)

In [7]:
ckpt = torch.load(str('/home/jovyan/dnalm/runs/annotation_bert_large_mane_no_intergenic_combined_42k_bpe_UNET_early_concat_atcg/bert_large_512_lastln_t2t_1000G_bs256_lr_1e-04_fp16/model_1750000/rmt_seglen_512_len42000_maxnsegm_10000_msz_5_bptt-1_lr5e-05_AdamW_constant_with_warmup_wd1e-04_p10000_bs_it50000/run_1/model_best/pytorch_model.bin'), map_location='cpu')
missing_k, unexpected_k = model_rmt.load_state_dict(ckpt, strict=False)
print(f'missing: {missing_k}') # if no missing tensors - that is correct, otherwise - no!
print(f'unexpected_k: {unexpected_k}') # if no missing tensors - that is correct, otherwise - no!

missing: []
unexpected_k: []


In [8]:
model_rmt = model_rmt.eval()

In [9]:
model_rmt.to("cuda")

RMTEncoderForLetterLevelTokenClassificationUNET(
  (model): BertForLetterLevelTokenClassification(
    (bert): BertModel(
      (embeddings): BertEmbeddings(
        (word_embeddings): Embedding(32005, 1024, padding_idx=3)
        (position_embeddings): Embedding(512, 1024)
        (token_type_embeddings): Embedding(2, 1024)
        (LayerNorm): LayerNorm((1024,), eps=1e-05, elementwise_affine=True)
        (dropout): Dropout(p=0.1, inplace=False)
      )
      (encoder): BertEncoder(
        (layer): ModuleList(
          (0-23): 24 x BertLayer(
            (pre_attention_ln): LayerNorm((1024,), eps=1e-05, elementwise_affine=True)
            (post_attention_ln): LayerNorm((1024,), eps=1e-05, elementwise_affine=True)
            (attention): BertAttention(
              (self): BertSelfAttention(
                (query): Linear(in_features=1024, out_features=1024, bias=True)
                (key): Linear(in_features=1024, out_features=1024, bias=True)
                (value): Linear(i

In [10]:
def prepare_sequence(sequence, tokenizer, max_length=2_500_000):
    model_input = dict()
    input_features = tokenizer([sequence], return_tensors='np')
    
#    print(input_features['input_ids'].shape)
    
    model_input['input_ids'] = input_features['input_ids']
    model_input['token_type_ids'] = input_features['token_type_ids']
    model_input['attention_mask'] = input_features['attention_mask']
    model_input['labels'] = np.random.randint(0, 5, (1, model_input['input_ids'].shape[1], 5)) # change it in future
    # model_input['labels_ohe'] = torch.randint(0, 6, (model_input['input_ids'].shape[0], 6)) # change it in future
    model_input['labels_mask'] = (input_features['input_ids'] > 5).astype(int)
    model_input['letter_level_attention_mask'] = np.expand_dims(np.ones(max_length).astype(int), axis=0) # it's okay because we will truncate paddings with mask
    model_input['letter_level_token_types_ids'] = np.expand_dims(np.zeros(max_length).astype(int), axis=0)
    
    token_repeater_numbers = []
    meaningful_tokens_only = model_input['input_ids'][model_input['input_ids'] > 5]
    for t in meaningful_tokens_only:
        atcg_seq_token = tokenizer.convert_ids_to_tokens(int(t))
        token_repeater_numbers.append(len(atcg_seq_token))
    
    token_repeater = []
    for n, i in enumerate(token_repeater_numbers):
        # print(i)
        for j in range(i):
            token_repeater.append(n)
    
    letter_level_tokens = []
    for letter in sequence:
        letter_level_tokens.append(tokenizer.convert_tokens_to_ids(letter))
        
    if len(letter_level_tokens) < max_length:
        letter_level_tokens += [-100] * (max_length - len(letter_level_tokens))
        token_repeater = token_repeater + [-100] * (max_length - len(token_repeater))
        
    model_input['letter_level_tokens'] = np.expand_dims(np.array(letter_level_tokens), axis=0)
    model_input['letter_level_labels'] = np.random.randint(0, 5, (1, max_length, 5)) # change it in future
    model_input['letter_level_labels_mask'] = model_input['letter_level_tokens'] != -100
    model_input['embedding_repeater'] = np.expand_dims(np.array(token_repeater), axis=0)
    
    for k, v in model_input.items():
        model_input[k] = torch.tensor(v)
        print(k, v.shape)
    
    return model_input

In [11]:
import random

# Define the length of the sequence and the nucleotides
sequence_length = 1000000
nucleotides = ['A', 'T', 'C', 'G']

# Generate a sequence with equal probability for each nucleotide
sequence = ''.join(random.choices(nucleotides, k=sequence_length))

In [12]:
def prediction(transcript_seq, model_rmt):
    
    seq = transcript_seq

    mode_input_data = prepare_sequence(seq, tokenizer)
    mode_input_data['input_ids'] = mode_input_data['input_ids'].to("cuda")
    mode_input_data['token_type_ids'] = mode_input_data['token_type_ids'].to("cuda")
    mode_input_data['attention_mask'] = mode_input_data['attention_mask'].to("cuda")
    mode_input_data['labels'] = mode_input_data['labels'].to("cuda")
    mode_input_data['labels_mask'] = mode_input_data['labels_mask'].to("cuda")
    mode_input_data['letter_level_attention_mask'] = mode_input_data['letter_level_attention_mask'].to("cuda")
    mode_input_data['letter_level_token_types_ids'] = mode_input_data['letter_level_token_types_ids'].to("cuda")

    mode_input_data['letter_level_tokens'] = mode_input_data['letter_level_tokens'].to("cuda")
    mode_input_data['letter_level_labels'] = mode_input_data['letter_level_labels'].to("cuda")
    mode_input_data['letter_level_labels_mask'] = mode_input_data['letter_level_labels_mask'].to("cuda")
    mode_input_data['embedding_repeater'] = mode_input_data['embedding_repeater'].to("cuda")
    mode_input_data['pos_weight'] = torch.tensor([1.0, 1.0, 1.0, 1.0, 1.0]).to("cuda")
    mode_input_data['pos_weight'] = mode_input_data['pos_weight'].repeat(1, len(transcript_seq), 1)

    with torch.no_grad():
        out = model_rmt(**mode_input_data)

    logits = out.logits[0, mode_input_data['letter_level_labels_mask'][0], :]
    logits = torch.sigmoid(logits)
    logits = logits.cpu()

    mode_input_data = prepare_sequence(seq, tokenizer)
    mode_input_data['input_ids'] = mode_input_data['input_ids'].to("cpu")
    mode_input_data['token_type_ids'] = mode_input_data['token_type_ids'].to("cpu")
    mode_input_data['attention_mask'] = mode_input_data['attention_mask'].to("cpu")
    mode_input_data['labels'] = mode_input_data['labels'].to("cpu")
    mode_input_data['labels_mask'] = mode_input_data['labels_mask'].to("cpu")
    mode_input_data['letter_level_attention_mask'] = mode_input_data['letter_level_attention_mask'].to("cpu")
    mode_input_data['letter_level_token_types_ids'] = mode_input_data['letter_level_token_types_ids'].to("cpu")

    mode_input_data['letter_level_tokens'] = mode_input_data['letter_level_tokens'].to("cpu")
    mode_input_data['letter_level_labels'] = mode_input_data['letter_level_labels'].to("cpu")
    mode_input_data['letter_level_labels_mask'] = mode_input_data['letter_level_labels_mask'].to("cpu")
    mode_input_data['embedding_repeater'] = mode_input_data['embedding_repeater'].to("cpu")
    mode_input_data['pos_weight'] = mode_input_data['pos_weight'].to("cpu")


    p_labels = np.array(logits)

    return p_labels

In [13]:
pred = prediction(sequence, model_rmt)

input_ids (1, 177559)
token_type_ids (1, 177559)
attention_mask (1, 177559)
labels (1, 177559, 5)
labels_mask (1, 177559)
letter_level_attention_mask (1, 2500000)
letter_level_token_types_ids (1, 2500000)
letter_level_tokens (1, 2500000)
letter_level_labels (1, 2500000, 5)
letter_level_labels_mask (1, 2500000)
embedding_repeater (1, 2500000)


/home/jovyan/dnalm/my_saved_conda_envs/gena/lib/python3.9/site-packages/transformers/modeling_utils.py:1101: FutureWarning: The `device` argument is deprecated and will be removed in v5 of Transformers.
  warnings.warn(


AttributeError: 'dict' object has no attribute 'logits'

# Configure model

In [2]:
os.chdir('/home/jovyan/dnalm/')

In [3]:
rmt_model_path = Path('/home/jovyan/dnalm/runs/annotation/bert_base_512_lastln_t2t_1000G_bs256_lr_1e-04_linear_fp16/model_2000000/rmt_seglen_512_len42000_maxnsegm_10000_msz_5_bptt-1_lr5e-05_AdamW_cosine_wd1e-04_p10_bs64_it50000/run_1/')

In [4]:
exp_config = json.load((rmt_model_path / 'config.json').open('r')) # it should be config.json that I sent you

In [5]:
for k in ['input_seq_len', 'model_cfg', 'model_cls', 'backbone_cls', 'input_size', 'num_mem_tokens', 'max_n_segments', 'tokenizer']:
    print(f'{k}: {exp_config[k]}')

input_seq_len: 42000
model_cfg: ./data/configs/L12-H768-A12-V32k-preln-lastln.json
model_cls: src.gena_lm.modeling_rmt:RMTEncoderForTokenClassification
backbone_cls: src.gena_lm.modeling_bert:BertForTokenClassification
input_size: 512
num_mem_tokens: 5
max_n_segments: 10000
tokenizer: ./data/tokenizers/t2t_1000h_multi_32k/


In [6]:
from transformers import AutoTokenizer, AutoConfig
tokenizer = AutoTokenizer.from_pretrained('./data/tokenizers/t2t_1000h_multi_32k/')

/home/jovyan/dnalm/my_saved_conda_envs/gena/lib/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [7]:
from src.gena_lm.modeling_bert import BertForLetterLevelTokenClassification, BertForTokenClassification

In [8]:
model_cfg = AutoConfig.from_pretrained('./data/configs/L24-H1024-A16-V32k-preln-lastln.json') # here it soulbe config for backbone model, don't change it, you can change only path to it
# model_cfg.num_labels = 6
# model_cfg.problem_type = 'multi_label_classification'
model_cls = BertForLetterLevelTokenClassification
model = model_cls(config=model_cfg)

AAAAAAAAAAAAAAAAAAAAAAAAAAAAA


In [9]:
model_cfg_small = AutoConfig.from_pretrained('./data/configs/L24-H1024-A16-V32k-preln-lastln.json') # here it soulbe config for backbone model, don't change it, you can change only path to it
model_cfg_small.num_labels = 6
model_cfg_small.problem_type = 'multi_label_classification'
model_cls_small = BertForTokenClassification
model_small = model_cls_small(config=model_cfg_small)

AAAAAAAAAAAAAAAAAAAAAAAAAAAAA


In [10]:
rmt_config = {
            'num_mem_tokens': exp_config['num_mem_tokens'],
            'max_n_segments': exp_config['max_n_segments'],
            'input_size': exp_config['input_size'],
            'bptt_depth': -1,
            'sum_loss': True,
            'tokenizer': tokenizer,
            'use_crf': False,
            'num_crf_classes': 6
        }
from src.gena_lm.modeling_rmt import RMTEncoderForLetterLevelTokenClassification
rmt_cls = RMTEncoderForLetterLevelTokenClassification
model_rmt = rmt_cls(model, model_small, **rmt_config)

In [11]:
# load pre-trained weights
# ckpt = torch.load(str('/home/jovyan/dnalm/runs/annotation/bert_base_512_lastln_t2t_1000G_bs256_lr_1e-04_linear_fp16/model_2000000/rmt_seglen_512_len4096_maxnsegm_10000_msz_5_bptt-1_lr5e-05_AdamW_cosine_wd1e-04_p10_bs_it50000/run_1/model_best/accelerate_state/pytorch_model.bin'), map_location='cpu')
ckpt = torch.load(str('/home/jovyan/dnalm/runs/annotation_2xlarge_letter_level_longer_training_full_length_seam_boost/bert_large_512_lastln_t2t_1000G_bs256_lr_1e-04_fp16/model_1750000/rmt_seglen_512_len42000_maxnsegm_10000_msz_5_bptt-1_lr1e-04_AdamW_constant_with_warmup_wd1e-04_p10000_bs_it500000/run_1/model_best/pytorch_model.bin'), map_location='cpu')

missing_k, unexpected_k = model_rmt.load_state_dict(ckpt, strict=False)
print(f'missing: {missing_k}') # if no missing tensors - that is correct, otherwise - no!
print(f'unexpected_k: {unexpected_k}') # if no missing tensors - that is correct, otherwise - no!

missing: []
unexpected_k: []


In [12]:
model_rmt = model_rmt.eval()

In [13]:
model_rmt.model.base_model == model_rmt.model.bert

True

In [14]:
model_rmt.model.base_model.embeddings

BertEmbeddings(
  (word_embeddings): Embedding(32005, 1024, padding_idx=3)
  (position_embeddings): Embedding(512, 1024)
  (token_type_embeddings): Embedding(2, 1024)
  (LayerNorm): LayerNorm((1024,), eps=1e-05, elementwise_affine=True)
  (dropout): Dropout(p=0.1, inplace=False)
)

In [15]:
model_rmt.model.embeddings

Embedding(32005, 1024, padding_idx=3)

In [16]:
model_rmt.sub_model.embeddings(torch.tensor([6, 8, 9, 15]))

tensor([[-0.0684,  0.0770,  0.1769,  ...,  0.0003,  0.1553,  0.0833],
        [-0.0807, -0.1065,  0.1723,  ..., -0.1466, -0.0674,  0.0789],
        [-0.1532, -0.0376, -0.0971,  ..., -0.1707,  0.0272,  0.1131],
        [-0.0671,  0.0177,  0.2218,  ...,  0.0841,  0.1391,  0.0947]],
       grad_fn=<EmbeddingBackward0>)

In [17]:
model_rmt.model.embeddings(torch.tensor([6, 8, 9, 15]))

tensor([[-0.0672,  0.0869,  0.1808,  ..., -0.0029,  0.1851,  0.0833],
        [-0.0754, -0.1099,  0.1772,  ..., -0.1418, -0.0609,  0.0784],
        [-0.1543, -0.0419, -0.0991,  ..., -0.1800,  0.0159,  0.1146],
        [-0.0707,  0.0312,  0.2198,  ...,  0.0883,  0.1581,  0.0992]],
       grad_fn=<EmbeddingBackward0>)

In [18]:
model_rmt.sub_model.embeddings(torch.tensor([[[18, 18, 18]]]))

tensor([[[[-0.0652,  0.0834, -0.0254,  ..., -0.1152,  0.1832,  0.1024],
          [-0.0652,  0.0834, -0.0254,  ..., -0.1152,  0.1832,  0.1024],
          [-0.0652,  0.0834, -0.0254,  ..., -0.1152,  0.1832,  0.1024]]]],
       grad_fn=<EmbeddingBackward0>)

In [66]:
model_rmt.sub_model.embeddings(torch.tensor([100]))

tensor([[-0.1066, -0.0106, -0.0432,  ...,  0.1169,  0.0392,  0.1195]],
       grad_fn=<EmbeddingBackward0>)

# Evaluation example

In [13]:
def prepare_sequence(sequence, tokenizer, max_length=250_000):
    model_input = dict()
    input_features = tokenizer(sequence, return_tensors='np')
    
    print(input_features['input_ids'].shape)
    
    model_input['input_ids'] = input_features['input_ids']
    model_input['token_type_ids'] = input_features['token_type_ids']
    model_input['attention_mask'] = input_features['attention_mask']
    model_input['labels'] = np.random.randint(0, 6, (1, model_input['input_ids'].shape[1], 6)) # change it in future
    # model_input['labels_ohe'] = torch.randint(0, 6, (model_input['input_ids'].shape[0], 6)) # change it in future
    model_input['labels_mask'] = (input_features['input_ids'] > 5).astype(int)
    model_input['letter_level_attention_mask'] = np.expand_dims(np.ones(max_length).astype(int), axis=0) # it's okay because we will truncate paddings with mask
    model_input['letter_level_token_types_ids'] = np.expand_dims(np.zeros(max_length).astype(int), axis=0)
    
    token_repeater_numbers = []
    meaningful_tokens_only = model_input['input_ids'][model_input['input_ids'] > 5]
    for t in meaningful_tokens_only:
        atcg_seq_token = tokenizer.convert_ids_to_tokens(int(t))
        # print(atcg_seq_token)
        token_repeater_numbers.append(len(atcg_seq_token))
    
    token_repeater = []
    for n, i in enumerate(token_repeater_numbers):
        # print(i)
        for j in range(i):
            token_repeater.append(n)
    
    letter_level_tokens = []
    for letter in sequence:
        # print(tokenizer.convert_tokens_to_ids(letter))
        letter_level_tokens.append(tokenizer.convert_tokens_to_ids(letter))
        
    if len(letter_level_tokens) < max_length:
        letter_level_tokens += [-100] * (max_length - len(letter_level_tokens))
        token_repeater = token_repeater + [-100] * (max_length - len(token_repeater))
        
    model_input['letter_level_tokens'] = np.expand_dims(np.array(letter_level_tokens), axis=0)
    model_input['letter_level_labels'] = np.random.randint(0, 6, (1, max_length, 6)) # change it in future
    model_input['letter_level_labels_mask'] = model_input['letter_level_tokens'] != -100
    model_input['embedding_repeater'] = np.expand_dims(np.array(token_repeater), axis=0)
    
    for k, v in model_input.items():
        model_input[k] = torch.tensor(v)
        print(k, v.shape)
    
    return model_input
    
    

In [14]:
artyom_seq = np.load('downstream_tasks/annotation/seq_25_06.npy')

In [15]:
mode_input_data = prepare_sequence(str(artyom_seq), tokenizer)

(1, 2313)
input_ids (1, 2313)
token_type_ids (1, 2313)
attention_mask (1, 2313)
labels (1, 2313, 6)
labels_mask (1, 2313)
letter_level_attention_mask (1, 250000)
letter_level_token_types_ids (1, 250000)
letter_level_tokens (1, 250000)
letter_level_labels (1, 250000, 6)
letter_level_labels_mask (1, 250000)
embedding_repeater (1, 250000)


In [16]:
with torch.no_grad():
    out = model_rmt(**mode_input_data)

/home/jovyan/dnalm/my_saved_conda_envs/gena/lib/python3.9/site-packages/transformers/modeling_utils.py:1052: FutureWarning: The `device` argument is deprecated and will be removed in v5 of Transformers.
  warnings.warn(


torch.Size([15048, 1, 6]) torch.Size([15048, 1])


/home/jovyan/dnalm/my_saved_conda_envs/gena/lib/python3.9/site-packages/torchcrf/__init__.py:249: UserWarning: where received a uint8 condition tensor. This behavior is deprecated and will be removed in a future version of PyTorch. Use a boolean condition instead. (Triggered internally at  ../aten/src/ATen/native/TensorCompare.cpp:328.)
  score = torch.where(mask[i].unsqueeze(1), next_score, score)


CRF loss: -62849.9375
CRF loss smoothed: 4.17663049697876


In [17]:
out.keys()

odict_keys(['loss', 'logits', 'decode'])

In [18]:
out['decode']

[[5,
  5,
  5,
  5,
  5,
  5,
  5,
  5,
  5,
  5,
  5,
  5,
  5,
  5,
  5,
  5,
  5,
  5,
  5,
  5,
  5,
  5,
  5,
  5,
  5,
  5,
  5,
  5,
  5,
  5,
  5,
  5,
  5,
  5,
  5,
  5,
  5,
  5,
  5,
  5,
  5,
  5,
  5,
  5,
  5,
  5,
  5,
  5,
  5,
  5,
  5,
  5,
  5,
  5,
  5,
  5,
  5,
  5,
  5,
  5,
  5,
  5,
  5,
  5,
  5,
  5,
  5,
  5,
  5,
  5,
  5,
  5,
  5,
  5,
  5,
  5,
  5,
  5,
  5,
  5,
  5,
  5,
  5,
  5,
  5,
  5,
  5,
  5,
  5,
  5,
  5,
  5,
  5,
  5,
  5,
  5,
  5,
  5,
  5,
  5,
  5,
  5,
  5,
  5,
  5,
  5,
  5,
  5,
  5,
  5,
  5,
  5,
  5,
  5,
  5,
  5,
  5,
  5,
  5,
  5,
  5,
  5,
  5,
  5,
  5,
  5,
  5,
  5,
  5,
  5,
  5,
  5,
  5,
  5,
  5,
  5,
  5,
  5,
  5,
  5,
  5,
  5,
  5,
  5,
  5,
  5,
  5,
  5,
  5,
  5,
  5,
  5,
  5,
  5,
  5,
  5,
  5,
  5,
  5,
  5,
  5,
  5,
  5,
  5,
  5,
  5,
  5,
  5,
  5,
  5,
  5,
  5,
  5,
  5,
  5,
  5,
  5,
  5,
  5,
  5,
  5,
  5,
  5,
  5,
  5,
  5,
  5,
  5,
  5,
  5,
  5,
  5,
  5,
  5,
  5,
  5,
  5,
  5,
  5,
  5,


In [21]:
out.logits.shape

torch.Size([1, 250000, 6])

In [22]:
out.logits

tensor([[[-4.7101, -4.5459, -4.6632, -5.0896, -6.5200,  7.1870],
         [-4.7304, -4.5681, -4.6298, -5.0996, -6.5198,  7.1690],
         [-4.7390, -4.5566, -4.6170, -5.0930, -6.5204,  7.1611],
         ...,
         [ 0.0000,  0.0000,  0.0000,  0.0000,  0.0000,  0.0000],
         [ 0.0000,  0.0000,  0.0000,  0.0000,  0.0000,  0.0000],
         [ 0.0000,  0.0000,  0.0000,  0.0000,  0.0000,  0.0000]]])

In [23]:
gena_predicts = torch.sigmoid(out.logits[0, mode_input_data['letter_level_labels_mask'][0], :])#[:, :4]

In [24]:
gena_predicts.shape

torch.Size([15048, 6])

In [25]:
gt = np.load('downstream_tasks/annotation/y_labels_25_06.npy')

In [26]:
# artyom_preds = np.load('downstream_tasks/annotation/p_labels.npy')

In [27]:
gena_predicts

tensor([[0.0089, 0.0105, 0.0093, 0.0061, 0.0015, 0.9992],
        [0.0087, 0.0103, 0.0097, 0.0061, 0.0015, 0.9992],
        [0.0087, 0.0104, 0.0098, 0.0061, 0.0015, 0.9992],
        ...,
        [0.1126, 0.0469, 0.8141, 0.0400, 0.0011, 0.7875],
        [0.1083, 0.0461, 0.8149, 0.0392, 0.0011, 0.7890],
        [0.1103, 0.0460, 0.8150, 0.0391, 0.0011, 0.7845]])

In [28]:
# artyom_preds.T

In [29]:
# np.all(gena_predicts == np.argmax(artyom_preds.T, axis=1))

In [30]:
# np.all(gt.T == 0)

In [30]:
from sklearn.metrics import average_precision_score


libgomp: Invalid value for environment variable OMP_NUM_THREADS


In [36]:
# label = 1
# y_label = gt.T[:, label]
# p_label = artyom_preds.T[:, label]

# print(average_precision_score(y_label, p_label, pos_label=1))

0.4710719735434303


In [42]:
label = 0
for label in range(6):
    y_label = gt.T[:, label]
    p_label = gena_predicts[:, label]

    print(average_precision_score(y_label, p_label, pos_label=1))

-0.0
0.4710708715509815
0.7393249711148424
-0.0
-0.0
0.7442971819002567


/home/jovyan/dnalm/my_saved_conda_envs/gena/lib/python3.9/site-packages/sklearn/metrics/_ranking.py:993: UserWarning: No positive class found in y_true, recall is set to one for all thresholds.
  warnings.warn(
/home/jovyan/dnalm/my_saved_conda_envs/gena/lib/python3.9/site-packages/sklearn/metrics/_ranking.py:993: UserWarning: No positive class found in y_true, recall is set to one for all thresholds.
  warnings.warn(
/home/jovyan/dnalm/my_saved_conda_envs/gena/lib/python3.9/site-packages/sklearn/metrics/_ranking.py:993: UserWarning: No positive class found in y_true, recall is set to one for all thresholds.
  warnings.warn(


# Testing

In [2]:
import h5py

In [38]:
dataset_file = h5py.File('/home/jovyan/dnalm/downstream_tasks/annotation/data/trans250k_token_atcg_val.hdf5', "r")

In [39]:
len(list(dataset_file.keys()))

11073

In [50]:
# seq around 10349000 on chr20
search_seq = 'TAGAGTAGTACAGTATGGACTACAGTTAATTTTATACAATTATGATTTAATACTTCACCTTTACATTTCTCTCAACTGCAAATAGGCCCATGTATGGTCTATATTTGTTTAAGTCTCATCAATTTTAACTTTTCATAGTAGATTTATATGTATTTTATGATAGTAAATAATAGACTAGACTAGTATCTACATATATTTTATCCATTCATGACATGCCTAATTTTTTCTAAATTTTCTCAATATTTCTAAGTAACACATTCATCTGCAAGTTTGTCTTAATTGTTGCAAATCTCCAAAGCATTTTCCAGTGTATTTATTGAAAAAAAATCCTGGCCAGGTACAATGGCACATGCCTGTAATCTCAGCATTTTGGGAGGCTGAGGAGGGAGGACTGCTTGAGCCCAGGAGTTTGAGACCAGCCTGGGCAACATAGTGAGATCTCATCTATACAAAAAATAGAAAAATTAGCCAGGCATGGTGGCATGCACCTGTGGTCTTAGCTATTTAGGAGGCTGAGATGGGAGGCTCAGTTGAGCCCAGGCAGTCAAGGGTGCAATGAGCTATGATCACACCGCTGTTCCCCAGCCTGGGCATCAGAGCAAGACCCTGTCTCAAAAAAAAGGAAGAAAGAAATTCACACAAAAGTGGATCCACACCATTCAAACCCATGATGTCCAAGGGTCCACTGTACATGCTCATGCTCATTAGCAATATTTTTTATCATCCAGGTTGGAGACGAGAAAGCTGAGCCCCCAAGATGTAATTTGTTCAAAGGGACACTGCTAATAAATTAGCCATGGTTTCAATCCTACAGTATAAGAAGTGTAATTCAACTTGGTCCAACAGATGTGAATGCCTACCACAGAAGTCATTGAGTGTCTTTCTAGGATAGAAAGATAAATCATATATAAGCCTACGCTCTCTGGTAGGGAAAATTATTAAAAATGAAAATTAAAAAAGCAATGATTCTGAAAAAATATATTCGTTCATTAAAAAAAAAAAATGAAAATTAAGCTTCTACGGTCACATTCAAATAGAGCCATAGCCGTCATTCTGGGAAGACAAAAGAAAGGATGGCCCATGCTGACCAAGGAGGTTTTAAGTGGTTTCACCCAAGAGGAAGCATTGGACCCGGACCCAGAAGGGTGAATGGATTTTTAAGGGGTAACCTGAGTTCTGTTTCTTTTCCCCCTTTTCTAGGCTGATTCCAACAAAACCAGAATTGATGAGGCCAACCAACGTGCAACAAAGATGCTGGGAAGTGGTTAAGTGTGCCCACCCGTGTTCTCCTCCAAATGCTGTCGGGCAAGATAGCTCCTTCATGCTTTTCTCATGGTATTATCTAGTAGGTCTGCACACATAACACACATCAGTCCACCCCCATTGTGAATGTTGTCCTGTGTCATCTGTCAGCTTCCCAACAATACTTTGTGTCTTTTGTTCTCTCTTGGTCTCTTTCTTTCCAAAGGTTGTACATAGTGGTCATTTGGTGGCTCTAACTCCTTGATGTCCTGAGTTTCATTTTTCATTTTCTCTCCTCGGTGGCATTTGCTGAATAACAACAATTTAGGAATGCTCAATGTGCTGTTGATTCTTTCAATCCACAGTATTGTTCTTGTAAAACTGTGACATTCCACAGAGTTACTGCCACGGTCCTTTGAGTGTCAGGCTCTGAATCTCTCAAAATGTGCCGTCTTTGGTTCCTCATGGCTGTTATCTGTCTTTATGATTTCATGATTAGACAATGTGGAATTACATAACAGGCATTGCACTAAAAGTGATGTGATTTATGCATTTATGCATGAGAACTAAATAGATTTTTAGATTCCTACTTAAACAAAAACTTTCCATGACAGTAGCATACTGATGAGACAACACACACACACACAAAACAACAGCAACAACAACAGAACAACAACAAAGCATGCTCAGTATTGAGACACTGTCAAGATTAAGTTATACCAGCAAAAGTGCAGTAGTGTCACTTTTTTCCTGTCAATATATAGAGACTTCTAAATCATAATCATCCTTTTTTAAAAAAAAGAATTTTAAAAAAGATGGATTTGACACACTCACCATTTAATCATTTCCAGCAAAATATATGTTTGGCTGAAATTATGTCAAATGGATGTAATATAGGGTTTGTTTGCTGCTTTTGATGGCTACGTTTTGGAGAGAGCAATCTTGCTGTGAAACAGTGTGGATGTAAATTTTATAAGGCTGACTCTTACTAACCACCATTTCCCCTGTGGTTTGTTATCAGTACAATTCTTTGTTGCTTAATCTAGAGCTATGCACACCAAATTGCTGAGATGTTTAGTAGCTGATAAAGAAACCTTTTAAAAAAATAATATAAATGAATGAAATATAAACTGTGAGATAAATATCATTATAGCATGT'

In [51]:
for i in range(len(list(dataset_file.keys()))):
    if search_seq in tokenizer.decode(np.array(dataset_file[f'transcript_{i}']["input_ids"])[0, 1:-1]):
        print(i)
        # break

6097
6098
6099
6100
6101
6102
6103
6104
6105
6106
6107
6108
6109
6110


In [92]:
idid = 6110
seq = tokenizer.decode(np.array(dataset_file[f'transcript_{idid}']["input_ids"])[0, 1:-1])
print(len(seq))
for i in range(len(seq)-10):
    if seq[i:i+10] == 'TTTTCTAGGC':
        print(i)

92472
89174


In [96]:
[1, 2] + [3]

[1, 2, 3]

In [93]:
print(seq[89300:89300+10])

TTCATGCTTT


In [86]:
np.array(dataset_file[f'transcript_{idid}']["labels"]).shape

(1, 92598, 6)

In [95]:
np.array(dataset_file[f'transcript_{idid}']["labels"])[0, 89300+5:89300+5+5, :]

array([[0, 1, 0, 1, 0, 0],
       [0, 1, 0, 1, 0, 0],
       [0, 1, 0, 1, 0, 0],
       [0, 1, 0, 1, 0, 0],
       [0, 1, 0, 1, 0, 0]], dtype=int8)

In [37]:
seq = tokenizer.decode(np.array(dataset_file[f'transcript_{81210}']["input_ids"])[0, 1:-1])
with open('sample_sample.fa', 'w') as f:
    f.write(seq)

In [29]:
len(list(dataset_file.keys()))

137864

In [37]:
from tqdm import tqdm
counter = 0
for i in tqdm(range(len(list(dataset_file.keys())))):
    counter += len(np.array(dataset_file[f'transcript_{i}']["input_ids"]).squeeze())
    if i == 80_000:
        break

 58%|█████▊    | 80000/137864 [01:50<01:19, 724.28it/s] 


In [38]:
counter

744805985

In [32]:
n = 1000
labels = np.array(dataset_file[f'transcript_{n}']["labels"]).squeeze()

In [33]:
labels.shape

(6, 5808)

In [24]:
128 ** 1.4

891.4437768152308

In [26]:
labels.T.tolist()[:]

[[0.0, 0.0, 1.0, 0.0, 0.0, 0.0],
 [0.0, 0.0, 1.0, 0.0, 0.0, 0.0],
 [0.0, 0.0, 1.0, 0.0, 0.0, 0.0],
 [0.0, 0.0, 1.0, 0.0, 0.0, 0.0],
 [0.0, 0.0, 1.0, 0.0, 0.0, 0.0],
 [0.0, 0.0, 1.0, 0.0, 0.0, 0.0],
 [0.0, 0.0, 1.0, 0.0, 0.0, 0.0],
 [0.0, 0.0, 1.0, 0.0, 0.0, 0.0],
 [0.0, 0.0, 1.0, 0.0, 0.0, 0.0],
 [0.0, 0.0, 1.0, 0.0, 0.0, 0.0],
 [0.0, 0.0, 1.0, 0.0, 0.0, 0.0],
 [0.0, 0.0, 1.0, 0.0, 0.0, 0.0],
 [0.0, 0.0, 1.0, 0.0, 0.0, 0.0],
 [0.0, 0.0, 1.0, 0.0, 0.0, 0.0],
 [0.0, 0.0, 1.0, 0.0, 0.0, 0.0],
 [0.0, 0.0, 1.0, 0.0, 0.0, 0.0],
 [0.0, 0.0, 1.0, 0.0, 0.0, 0.0],
 [0.0, 0.0, 1.0, 0.0, 0.0, 0.0],
 [0.0, 0.0, 1.0, 0.0, 0.0, 0.0],
 [0.0, 0.0, 1.0, 0.0, 0.0, 0.0],
 [0.0, 0.0, 1.0, 0.0, 0.0, 0.0],
 [0.0, 0.0, 1.0, 0.0, 0.0, 0.0],
 [0.0, 0.0, 1.0, 0.0, 0.0, 0.0],
 [0.0, 0.0, 1.0, 0.0, 0.0, 0.0],
 [0.0, 0.0, 1.0, 0.0, 0.0, 0.0],
 [0.0, 0.0, 1.0, 0.0, 0.0, 0.0],
 [0.0, 0.0, 1.0, 0.0, 0.0, 0.0],
 [0.0, 0.0, 1.0, 0.0, 0.0, 0.0],
 [0.0, 0.0, 1.0, 0.0, 0.0, 0.0],
 [0.0, 0.0, 1.0, 0.0, 0.0, 0.0],
 [0.0, 1.0

In [46]:
(input_ids != 1) & (input_ids != 2)

array([False,  True,  True, ...,  True,  True, False])

In [47]:
input_ids = input_ids[(input_ids != 1) & (input_ids != 2)]
input_ids.shape

(1099,)

In [48]:
atcg_seq = ''
for t in input_ids[:4096]:
    atcg_seq_token = tokenizer.convert_ids_to_tokens(int(t))
    atcg_seq += atcg_seq_token
    
print(len(atcg_seq))

6704


# Get data from dataset

In [13]:
from transcript_GenePredictionDataset_letter_level import AnnotationDataset

In [14]:
dtst = AnnotationDataset('/home/jovyan/dnalm/downstream_tasks/annotation/data/trans250k_token_atcg_val.hdf5', 4096, 40000, tokenizer)

In [15]:
len(dtst)

11073

In [24]:
combined_predictions_torch = torch.cat(combined_predictions, axis=0)

In [25]:
combined_predictions_torch.shape

torch.Size([237148714, 6])

In [27]:
combined_y_torch = torch.tensor(np.concatenate(combined_y, axis=0))

In [28]:
combined_y_torch.shape

torch.Size([237148714, 6])

In [31]:
for label in range(6):
    y_label = combined_y_torch[:, label]
    p_label = combined_predictions_torch[:, label]

    print(f'class id: {label}', average_precision_score(y_label, p_label, pos_label=1))

class id: 0 0.18313583774675593
class id: 1 0.5411635983484983
class id: 2 0.9467677049104779
class id: 3 0.34300691924655574
class id: 4 0.6514042904817461
class id: 5 0.8003978242356212


In [22]:
combined_predictions, combined_y = [], []
for num, model_data in enumerate(dtst):
    with torch.no_grad():
        
        model_data_mod = dict()
        val_dataset_sequence = None
        for k, v in model_data.items():
            if k != 'labels_ohe':
                model_data_mod[k] = torch.tensor(np.expand_dims(v, axis=0))
                # print(k, model_data_mod[k].shape)
                
        out = model_rmt(**model_data_mod)
        gena_predicts = torch.sigmoid(out.logits[0, model_data['letter_level_labels_mask'], :])
        combined_predictions.append(gena_predicts)
        combined_y.append(model_data['letter_level_labels'][model_data['letter_level_labels_mask'], :])
        print(combined_predictions[-1].shape, combined_y[-1].shape, num)

torch.Size([19208, 6]) (19208, 6) 0
torch.Size([19208, 6]) (19208, 6) 1
torch.Size([19145, 6]) (19145, 6) 2
torch.Size([19145, 6]) (19145, 6) 3
torch.Size([18888, 6]) (18888, 6) 4
torch.Size([18888, 6]) (18888, 6) 5
torch.Size([18888, 6]) (18888, 6) 6
torch.Size([13091, 6]) (13091, 6) 7
torch.Size([10517, 6]) (10517, 6) 8
torch.Size([7177, 6]) (7177, 6) 9
torch.Size([6936, 6]) (6936, 6) 10
torch.Size([6936, 6]) (6936, 6) 11
torch.Size([11246, 6]) (11246, 6) 12
torch.Size([25713, 6]) (25713, 6) 13
torch.Size([25713, 6]) (25713, 6) 14
torch.Size([25713, 6]) (25713, 6) 15
torch.Size([25713, 6]) (25713, 6) 16
torch.Size([25711, 6]) (25711, 6) 17
torch.Size([25674, 6]) (25674, 6) 18
torch.Size([25662, 6]) (25662, 6) 19
torch.Size([25729, 6]) (25729, 6) 20
torch.Size([25729, 6]) (25729, 6) 21
torch.Size([25559, 6]) (25559, 6) 22
torch.Size([25559, 6]) (25559, 6) 23
torch.Size([25559, 6]) (25559, 6) 24
torch.Size([25822, 6]) (25822, 6) 25
torch.Size([25747, 6]) (25747, 6) 26
torch.Size([25766

AssertionError: 

In [163]:
model_data = dict()
val_dataset_sequence = None
for k, v in dtst[1000].items():
    if k != 'labels_ohe' and k != 'seq':
        model_data[k] = torch.tensor(np.expand_dims(v, axis=0))
        print(k, model_data[k].shape)
    if k == 'seq':
        val_dataset_sequence = v
    # if k == 'letter_level_labels':
    #     model_data[k] = torch.randint(0, 2, (1, 50000, 6))

input_ids torch.Size([1, 5096])
token_type_ids torch.Size([1, 5096])
attention_mask torch.Size([1, 5096])
labels torch.Size([1, 5096, 6])
labels_mask torch.Size([1, 5096])
letter_level_tokens torch.Size([1, 50000])
letter_level_labels torch.Size([1, 50000, 6])
letter_level_labels_mask torch.Size([1, 50000])
embedding_repeater torch.Size([1, 50000])
letter_level_attention_mask torch.Size([1, 50000])
letter_level_token_types_ids torch.Size([1, 50000])


In [164]:
with open('demo_seq_for_artyom.txt', 'w') as f:
    f.write(val_dataset_sequence)

In [165]:
# mode_input_data['input_ids'][0, :10]
# mode_input_data['token_type_ids'] # change to float
# mode_input_data['attention_mask'] 
# mode_input_data['labels_mask']
# mode_input_data['letter_level_tokens']
# mode_input_data['letter_level_labels_mask']
mode_input_data['letter_level_attention_mask']

tensor([[1, 1, 1,  ..., 1, 1, 1]])

In [166]:
# model_data['input_ids'][0, :10]
# model_data['token_type_ids']
# model_data['attention_mask']
# model_data['labels_mask']
# model_data['letter_level_tokens']
# model_data['letter_level_labels_mask']
model_data['letter_level_attention_mask']

tensor([[1, 1, 1,  ..., 1, 1, 1]])

In [167]:
with torch.no_grad():
    out = model_rmt(**model_data)

/home/jovyan/dnalm/my_saved_conda_envs/gena/lib/python3.9/site-packages/transformers/modeling_utils.py:1052: FutureWarning: The `device` argument is deprecated and will be removed in v5 of Transformers.
  warnings.warn(


In [168]:
gena_predicts = torch.sigmoid(out.logits[0, model_data['letter_level_labels_mask'][0], :])
# gena_predicts = torch.sigmoid(out.logits[0, mode_input_data['letter_level_labels_mask'][0], :])

In [169]:
gena_predicts

tensor([[0.0094, 0.0047, 0.0045, 0.0050, 0.0038, 0.9996],
        [0.0095, 0.0046, 0.0046, 0.0050, 0.0038, 0.9996],
        [0.0094, 0.0047, 0.0047, 0.0050, 0.0038, 0.9996],
        ...,
        [0.0319, 0.0092, 0.0037, 0.0133, 0.0071, 0.9997],
        [0.0319, 0.0094, 0.0038, 0.0132, 0.0072, 0.9997],
        [0.0334, 0.0094, 0.0040, 0.0150, 0.0067, 0.9997]])

In [170]:
artyom_preds_2 = np.load('downstream_tasks/annotation/p_labels_2.npy')
artyom_preds_2.T

array([[0.00839381, 0.00414679, 0.00652329, 0.00174201],
       [0.00904063, 0.00466054, 0.0053945 , 0.001471  ],
       [0.00900883, 0.00462937, 0.00548028, 0.00144254],
       ...,
       [0.04796838, 0.0556737 , 0.07997473, 0.01061047],
       [0.04693732, 0.05667868, 0.07574777, 0.01149839],
       [0.04645827, 0.05599522, 0.07761885, 0.01129771]], dtype=float32)

In [171]:
for i in range(4):
    label = i
    y_label = model_data['letter_level_labels'][0, model_data['letter_level_labels_mask'][0], :][:, label]
    # y_label = mode_input_data['letter_level_labels'][0, mode_input_data['letter_level_labels_mask'][0], :][:, label]
    
    p_label = gena_predicts[:, label] #artyom_preds_2.T[:, label] 

    print(average_precision_score(y_label, p_label))

0.09737207770568622
0.4011501433564016
0.8436340848208547
0.1100042716721536


In [51]:
model_data['letter_level_labels_mask'].shape

torch.Size([1, 40000])